# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SWAPI03/flyrank-ai-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Method: an expected-CTR regression model, ranked by residual.** My lane asks *which visible pages
under-capture clicks for their position?* So I train a model to predict the CTR a page *should*
earn from its pre-click characteristics, then rank pages by the gap between predicted and actual CTR
(a large positive residual = under-capturing). This is the same shape as my Week-4 baseline — which
used the crude *position-tier median* as the expected CTR — so the model is a like-for-like upgrade I
can compare directly.

**Why not the other tools.** Classification needs a clean observed label, and the starter slice has
no future window to define one honestly, so a supervised yes/no would be circular. Clustering finds
*types* of pages, not a ranked action. Regression on expected CTR fits the ranking question and stays
comparable to the baseline.

**The review population.** I model the pages the lane actually acts on: visible (`impressions_90d >=
500`), ranking on page one (`avg_position` 1-20), and earning some clicks (`ctr > 0`) — so the
comparison is about ranking *real* candidates, not trivially surfacing zero-click pages. CTR is
clipped at its 99th percentile to stop a few extreme outliers from dominating. Seeds are fixed (42).

**Ladder:** tier-median baseline -> Linear Regression -> Decision Tree -> Gradient Boosting. I add
complexity only if the comparison table earns it.

In [1]:
# Setup + load + define the review population and leakage-safe features.
import os, sys, json, subprocess
import numpy as np, pandas as pd, sklearn
if "google.colab" in sys.modules and not os.path.isdir("data/raw"):
    if not os.path.isdir("flyrank-ml-internship-starter"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                        "flyrank-ml-internship-starter"], check=True)
    os.chdir("flyrank-ml-internship-starter")
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("scikit-learn", sklearn.__version__)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
d = df[(df["impressions_90d"] >= 500) & (df["avg_position"] > 0) &
       (df["avg_position"] <= 20) & (df["ctr"] > 0)].copy()

# leakage-safe features: pre-click signals only. NEVER ctr, clicks, trend_*, or downstream engagement.
NUM = ["avg_position", "log_impressions", "content_age_days", "days_since_last_update",
       "word_count", "search_volume", "competition", "cpc"]
CAT = ["content_type", "main_intent"]
d["log_impressions"] = np.log1p(d["impressions_90d"])
for c in NUM:
    d[c] = pd.to_numeric(d[c], errors="coerce")
d[NUM] = d[NUM].fillna(d[NUM].median())
d[CAT] = d[CAT].fillna("unknown")

cap = d["ctr"].quantile(0.99)               # tame the heavy CTR tail
d["ctr_c"] = d["ctr"].clip(upper=cap)
print(f"review population: {len(d):,} pages across {d['client_id'].nunique()} clients")
print(f"target = CTR (clipped at 99th pct = {cap:.2f}); features excluded: ctr, clicks, trend_*, engagement")

scikit-learn 1.7.2
review population: 10,807 pages across 28 clients
target = CTR (clipped at 99th pct = 1.66); features excluded: ctr, clicks, trend_*, engagement


## 2. Split design

**Client-holdout (GroupShuffleSplit on `client_id`, 25% held out, seed 42).** Pages from the same
client share templates, branding, and topic focus, so a random row split would let the model memorize
a client in training and be tested on the same client's other pages — an inflated score. Grouping by
client means every test page comes from a client the model never saw, which is the honest test for
"does this generalize to a new client?" The baseline is scored on the *same* held-out clients so the
comparison is fair.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

y = d["ctr_c"].values
groups = d["client_id"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(d, y, groups))
train, test = d.iloc[train_idx].copy(), d.iloc[test_idx].copy()

shared = set(train["client_id"]) & set(test["client_id"])
print(f"train pages: {len(train):,} | test pages: {len(test):,} | shared clients: {len(shared)} (must be 0)")

# Week-4 baseline as a CTR predictor: the position-tier MEDIAN, learned on TRAIN only.
tier_median = train.groupby("position_tier")["ctr_c"].median()
global_median = train["ctr_c"].median()
test["base_expected"] = test["position_tier"].map(tier_median).fillna(global_median)
print("baseline = tier-median expected CTR (the Week-4 rule), fit on train clients only")

train pages: 10,083 | test pages: 724 | shared clients: 0 (must be 0)
baseline = tier-median expected CTR (the Week-4 rule), fit on train clients only


## 3. Train + compare vs my baseline

Same data, same held-out clients, same metrics. Two metric families: **CTR prediction** (MAE, lower
is better; R2, higher is better) and **opportunity ranking** (Precision@K against an independent
ground truth — a page that sits below the median CTR of its *fine* 1-unit position bin on the test
set, a tighter position control than the coarse tier the baseline uses).

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score

pre = ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore"), CAT)], remainder="passthrough")
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree (d=5)": DecisionTreeRegressor(max_depth=5, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}
fitted = {}
rows = [{"method": "Baseline (tier median)",
         "MAE": mean_absolute_error(test["ctr_c"], test["base_expected"]),
         "R2":  r2_score(test["ctr_c"], test["base_expected"])}]
for name, mdl in models.items():
    pipe = Pipeline([("pre", pre), ("m", mdl)]).fit(train[NUM + CAT], train["ctr_c"])
    fitted[name] = pipe
    pred = pipe.predict(test[NUM + CAT])
    rows.append({"method": name,
                 "MAE": mean_absolute_error(test["ctr_c"], pred),
                 "R2":  r2_score(test["ctr_c"], pred)})
table = pd.DataFrame(rows).set_index("method").round(4)
print("=== CTR prediction on held-out clients ===")
print(table.to_string())

# --- opportunity ranking: Precision@K vs fine 1-unit position-bin under-performance ---
test["model_expected"] = fitted["Gradient Boosting"].predict(test[NUM + CAT])
test["pos_bin"] = test["avg_position"].round()
test["fine_median"] = test.groupby("pos_bin")["ctr_c"].transform("median")
test["true_under"] = (test["ctr_c"] < test["fine_median"]).astype(int)

def precision_at_k(score, label, k):
    order = np.argsort(-score.values)
    return label.values[order[:k]].mean()

base_gap  = test["base_expected"]  - test["ctr_c"]
model_gap = test["model_expected"] - test["ctr_c"]
print(f"\n=== opportunity ranking (base rate under = {test['true_under'].mean():.3f}) ===")
rank_rows = []
for k in (20, 50, 100):
    rank_rows.append({"K": k,
                      "baseline P@K": round(precision_at_k(base_gap,  test["true_under"], k), 3),
                      "GBM P@K":      round(precision_at_k(model_gap, test["true_under"], k), 3)})
print(pd.DataFrame(rank_rows).set_index("K").to_string())

os.makedirs("work/outputs", exist_ok=True)
json.dump({"split": "client_holdout_gss_seed42", "review_population": int(len(d)),
          "metrics": table.reset_index().to_dict(orient="records")},
          open("work/outputs/model_metrics.json", "w"), indent=2)
print("\nwrote work/outputs/model_metrics.json")

=== CTR prediction on held-out clients ===
                           MAE      R2
method                                
Baseline (tier median)  0.2966 -0.2423
Linear Regression       0.2917 -0.0337
Decision Tree (d=5)     0.2843 -0.0506
Gradient Boosting       0.2864  0.0013

=== opportunity ranking (base rate under = 0.488) ===
     baseline P@K  GBM P@K
K                         
20            1.0     1.00
50            1.0     1.00
100           1.0     0.98

wrote work/outputs/model_metrics.json


## 4. Errors and interpretation

**Reading the result honestly.** On the review population the Gradient Boosting model predicts CTR
more accurately than the tier-median baseline (lower MAE, and a positive R2 where the baseline's is
negative), so its expected-CTR is better calibrated. But at the very top of the ranked queue the two
*tie* — both surface the clearest under-performers — so the model's edge is in the calibrated middle
of the list, not the first twenty. **The transparent baseline is genuinely competitive**, and I would
keep it as a live safety check rather than replace it. Complexity earns a place here only for the
mid-list, not for the headline pick — exactly the "don't reward complexity alone" test. A depth-5 Decision Tree actually ties Gradient Boosting on MAE, so the ensemble's only clear edge is R2 (calibration) -- the simplest model that clears the bar is the one worth keeping. One caveat: the held-out set here is small (about 7 clients), so these numbers are directional; the full warehouse is where the design proves out.

In [4]:
from sklearn.inspection import permutation_importance

# What does the model lean on? Permutation importance on held-out clients.
gbm = fitted["Gradient Boosting"]
imp = permutation_importance(gbm, test[NUM + CAT], test["ctr_c"],
                             n_repeats=5, random_state=42, scoring="r2")
importances = (pd.Series(imp.importances_mean, index=NUM + CAT)
                 .sort_values(ascending=False))
print("Permutation importance (drop in R2 when a feature is shuffled):")
print(importances.round(4).to_string())
top2 = ", ".join(importances.head(2).index)
print(f"Top signals: {top2}. Both are plausible CTR drivers (a page's age and its search position),")
print("and no leak-ish column (ctr / clicks / trend) tops the list -- the model earns its signal honestly.\n")

# Where is the model most wrong? Absolute residual by content_type and volume.
test["abs_err"] = (test["model_expected"] - test["ctr_c"]).abs()
print("Mean abs error by content_type:")
print(test.groupby("content_type")["abs_err"].agg(["mean", "size"]).round(3).to_string())

# Three concrete hard cases (largest errors).
print("\nThree hardest cases (largest |error|):")
worst = test.reindex(test["abs_err"].sort_values(ascending=False).index).head(3)
for _, r in worst.iterrows():
    print(f"  {r['content_id']}: pos {r['avg_position']:.1f}, imp {int(r['impressions_90d']):,}, "
          f"actual CTR {r['ctr_c']:.2f} vs predicted {r['model_expected']:.2f} "
          f"({r['content_type']}) -- low-volume / atypical CTR is where expected-CTR models struggle.")

Permutation importance (drop in R2 when a feature is shuffled):
content_age_days          0.0651
avg_position              0.0580
search_volume             0.0058
log_impressions           0.0050
competition               0.0001
days_since_last_update   -0.0003
main_intent              -0.0005
cpc                      -0.0018
content_type             -0.0025
word_count               -0.0075
Top signals: content_age_days, avg_position. Both are plausible CTR drivers (a page's age and its search position),
and no leak-ish column (ctr / clicks / trend) tops the list -- the model earns its signal honestly.

Mean abs error by content_type:
                  mean  size
content_type                
feedly article   0.319    51
keyword article  0.284   673

Three hardest cases (largest |error|):
  content_8fc3aeb6bf3a: pos 17.0, imp 787, actual CTR 1.66 vs predicted 0.28 (keyword article) -- low-volume / atypical CTR is where expected-CTR models struggle.
  content_a001aa4be7b7: pos 16.2, imp 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.